# **Home Mortgage Application Approval Classifier - Final Demo**

This notebook loads the Optuna-tuned XGBoost from `artifacts/`, demonstrates the full modelling-to-explainability pipeline on 3 representative test applications, builds a Platt-calibrated version, persists the calibrated model, reloads it, and compares calibration before showing SHAP global and local interpretability.

All figures are written to `figures/final_demo/`. A narrative summary is written to `markdown/final_demo_summary.md`.

**Scope:** decision-support demonstration only - not an autonomous approval/rejection system. Uses the same artifact and protocol as Step 8 (Calibration & SHAP) and Step 9 (Fairness): `scale_pos_weight=0.3396` fixed, 78-feature canonical schema, `tract_minority_population_percent` flagged `is_race_proxy=1`.

## **Step 0 - Setup & pre-flight**

Imports, paths, and a standing column-order guard that asserts the array column order exactly matches `day7_model_features.csv`. Loads the tuned model from `artifacts/hmda_tuned_xgboost.pkl` (`.json` booster fallback) and reproduces the Step 7 hold-out metrics as proof the correct artifact was loaded. Nothing is trained in this step.

In [1]:
import os, time, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import joblib, xgboost as xgb
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, confusion_matrix, roc_curve,
                             precision_recall_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
%matplotlib inline
import matplotlib.pyplot as plt
import shap
warnings.filterwarnings('ignore')

ROOT = Path('/Volumes/Mitul/Projects/home-mortage-approval-predictor')
M = ROOT / 'data' / 'processed' / 'modelling'
ART = ROOT / 'artifacts'
FIG = ROOT / 'figures' / 'final_demo'; FIG.mkdir(parents=True, exist_ok=True)
MARK = ROOT / 'markdown'; MARK.mkdir(exist_ok=True)

print('ROOT', ROOT)
print('xgboost', xgb.__version__, '| shap', shap.__version__)

# Column-order guard
feat = pd.read_csv(M / 'day7_model_features.csv')
assert len(feat) == 78, f"expected 78 features, got {len(feat)}"
FEATURES = feat['feature'].tolist()
assert set(FEATURES).issubset(set(pd.read_parquet(M / 'X_val.parquet').columns))
print('loaded', len(FEATURES), 'model features | race-proxy flagged:', int(feat['is_race_proxy'].sum()))

Xval = pd.read_parquet(M / 'X_val.parquet').astype(np.float32)
yval = pd.read_parquet(M / 'y_val.parquet')['approved'].astype('int8').values
Xtest = pd.read_parquet(M / 'X_test.parquet').astype(np.float32)
ytest = pd.read_parquet(M / 'y_test.parquet')['approved'].astype('int8').values
Xv, Xt = Xval[FEATURES].values, Xtest[FEATURES].values
assert list(Xval[FEATURES].columns) == FEATURES, 'FEATURE column order mismatch vs day7_model_features.csv'
print('alignment guard passed: array order == CSV order')

# Load the tuned model
model = None
try:
    model = joblib.load(ART / 'hmda_tuned_xgboost.pkl')
    print('loaded model from .pkl ->', type(model).__name__)
except Exception as e:
    print('pkl load failed:', repr(e))
    bst = xgb.Booster(); bst.load_model(ART / 'hmda_tuned_xgboost.json'); model = bst
    print('loaded model from .json -> Booster')

def raw_proba(X):
    if isinstance(model, xgb.Booster):
        return model.predict(xgb.DMatrix(X))
    return model.predict_proba(X)[:, 1]

p_val = raw_proba(Xv)
p_test = raw_proba(Xt)
print('REPRODUCED  val  ROC-AUC=%.4f PR-AUC=%.4f' % (roc_auc_score(yval, p_val), average_precision_score(yval, p_val)))
print('REPRODUCED test  ROC-AUC=%.4f PR-AUC=%.4f' % (roc_auc_score(ytest, p_test), average_precision_score(ytest, p_test)))

/Volumes/Mitul/Projects/home-mortage-approval-predictor/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT /Volumes/Mitul/Projects/home-mortage-approval-predictor
xgboost 3.4.1 | shap 0.52.0
loaded 78 model features | race-proxy flagged: 1
alignment guard passed: array order == CSV order
loaded model from .pkl -> XGBClassifier
REPRODUCED  val  ROC-AUC=0.8930 PR-AUC=0.9531
REPRODUCED test  ROC-AUC=0.8933 PR-AUC=0.9532


## **Step 1 - Select 3 representative test applications**

Draws 3 cases from the **test set** using the **calibrated** probabilities (Platt scaling fit on `X_val`), so the selection reflects the business-facing output:

- **Highest P(approve)** - near-certain approval (high chance of approval)
- **Lowest P(approve)** - near-certain denial (low predicted probability)
- **Nearest the business threshold (0.858)** - borderline, just above the threshold

In [2]:
# Fit Platt calibrator on X_val only (same as Step 8)
cal = LogisticRegression()
cal.fit(p_val.reshape(-1, 1), yval)
pc_val = cal.predict_proba(p_val.reshape(-1, 1))[:, 1]
pc_test = cal.predict_proba(p_test.reshape(-1, 1))[:, 1]

THRESH = 0.858  # business threshold (precision-weighted Fbeta=0.3 from Step 8)

# Select 3 representative test points on the CALIBRATED scale
hi_app = int(np.argmax(pc_test))
hi_den = int(np.argmin(pc_test))
border = int(np.argmin(np.abs(pc_test - THRESH)))

print('Selected test indices:')
print('  high-approval  :', hi_app, '(calibrated P(approve) = %.4f)' % pc_test[hi_app])
print('  denied (low)   :', hi_den, '(calibrated P(approve) = %.4f)' % pc_test[hi_den])
print('  borderline     :', border, '(calibrated P(approve) = %.4f, threshold=%.3f)' % (pc_test[border], THRESH))
print()
print('Actual approved :', ytest[hi_app], ytest[hi_den], ytest[border])

Selected test indices:
  high-approval  : 384459 (calibrated P(approve) = 0.9832)
  denied (low)   : 409154 (calibrated P(approve) = 0.1014)
  borderline     : 658691 (calibrated P(approve) = 0.8580, threshold=0.858)

Actual approved : 1 0 1


## **Step 2 - Uncalibrated model: full-test metrics + per-application display**

Shows the overall test-set metrics (ROC-AUC, PR-AUC, precision/recall/F1) and the confusion matrix at the business threshold for the **raw (uncalibrated)** model. Then prints each of the 3 selected applications with calibrated P(approve), denial risk, risk tier, and top SHAP factors. The raw model is retained for SHAP (logistic calibration is monotonic, preserving ranking and SHAP additivity).

In [3]:
# --- Full test-set metrics (uncalibrated) ---
def metrics_table(y, p, name):
    pred = (p >= 0.5).astype(int)
    roc = roc_auc_score(y, p)
    prauc = average_precision_score(y, p)
    prec = precision_score(y, pred)
    rec = recall_score(y, pred)
    f1 = f1_score(y, pred)
    print('[%s @0.5] ROC-AUC=%.4f PR-AUC=%.4f P=%.4f R=%.4f F1=%.4f' % (name, roc, prauc, prec, rec, f1))
    return dict(roc_auc=roc, pr_auc=prauc, precision=prec, recall=rec, f1=f1)

print('=== UNCALIBRATED MODEL ===')
m_test = metrics_table(ytest, p_test, 'X_test')
m_val = metrics_table(yval, p_val, 'X_val')

# Confusion matrix at business threshold (uncalibrated)
pred_test_raw = (p_test >= THRESH).astype(int)
cm = confusion_matrix(ytest, pred_test_raw)
tn, fp, fn, tp = cm.ravel()
print()
print('Confusion matrix @ THRESH=%.3f (uncalibrated, X_test):' % THRESH)
print('  TN=%d  FP=%d  FN=%d  TP=%d' % (tn, fp, fn, tp))
print('  TPR=%.4f  FPR=%.4f  FNR=%.4f' % (tp/(tp+fn), fp/(fp+tn), fn/(fn+tp)))

# --- Per-application display (calibrated) ---
def risk_tier(p):
    if p < 0.20: return 'Low (high approval prob)'
    if p < 0.50: return 'Moderate'
    if p < 0.75: return 'High'
    return 'Very high (low approval prob)'

print()
print('=== Per-application summary (calibrated probabilities) ===')
for label, i in [('HIGH APPROVAL', hi_app), ('DENIED (low)', hi_den), ('BORDERLINE', border)]:
    print()
    print('[%s] index=%d' % (label, i))
    print('  Calibrated P(approve) = %.4f | denial risk = %.4f | tier: %s' % (pc_test[i], 1-pc_test[i], risk_tier(1-pc_test[i])))
    print('  Actual approved = %d' % ytest[i])

=== UNCALIBRATED MODEL ===
[X_test @0.5] ROC-AUC=0.8933 PR-AUC=0.9532 P=0.9143 R=0.8571 F1=0.8848
[X_val @0.5] ROC-AUC=0.8930 PR-AUC=0.9531 P=0.9142 R=0.8565 F1=0.8844

Confusion matrix @ THRESH=0.858 (uncalibrated, X_test):
  TN=407337  FP=11668  FN=807112  TP=426880
  TPR=0.3459  FPR=0.0278  FNR=0.6541

=== Per-application summary (calibrated probabilities) ===

[HIGH APPROVAL] index=384459
  Calibrated P(approve) = 0.9832 | denial risk = 0.0168 | tier: Low (high approval prob)
  Actual approved = 1

[DENIED (low)] index=409154
  Calibrated P(approve) = 0.1014 | denial risk = 0.8986 | tier: Very high (low approval prob)
  Actual approved = 0

[BORDERLINE] index=658691
  Calibrated P(approve) = 0.8580 | denial risk = 0.1420 | tier: Low (high approval prob)
  Actual approved = 1


## **Step 3 - Build, persist & reload the calibrated model**

Fits the Platt (logistic) scaler on `X_val` only - exactly as in Step 8 - but now **persists** it to `artifacts/calibrated_model.pkl`. The calibrated model is then reloaded from disk and used to re-predict on the same 3 applications. This demonstrates that the calibrator survives a save/reload cycle (important because retraining the scaler takes ~16 min on this system).

In [4]:
# Persist the calibrated model (LogisticRegression Platt scaler)
CAL_MODEL_PATH = ART / 'calibrated_model.pkl'
joblib.dump(cal, CAL_MODEL_PATH)
print('persisted calibrated model ->', CAL_MODEL_PATH)
print('  coef_ = %.6f, intercept_ = %.6f' % (cal.coef_[0][0], cal.intercept_[0]))

# Reload from disk
cal_loaded = joblib.load(CAL_MODEL_PATH)
print('reloaded calibrated model ->', type(cal_loaded).__name__)
print('  coef_ match:', np.allclose(cal.coef_, cal_loaded.coef_) and np.allclose(cal.intercept_, cal_loaded.intercept_))

persisted calibrated model -> /Volumes/Mitul/Projects/home-mortage-approval-predictor/artifacts/calibrated_model.pkl
  coef_ = 6.253311, intercept_ = -2.182117
reloaded calibrated model -> LogisticRegression
  coef_ match: True


## **Step 4 - Compare calibrated vs uncalibrated: metrics & the 3 points**

Re-runs the full test-set evaluation with the **loaded** calibrated model and prints:

- ROC-AUC, PR-AUC, precision/recall/F1 at 0.5 and at the business threshold
- Confusion matrix at the business threshold
- Side-by-side comparison of the 3 selected applications under both raw and calibrated probabilities
- Calibration curve (reliability diagram) showing the improvement

In [5]:
# --- Full test-set metrics (calibrated, loaded) ---
pc_test_loaded = cal_loaded.predict_proba(p_test.reshape(-1, 1))[:, 1]
pc_val_loaded = cal_loaded.predict_proba(p_val.reshape(-1, 1))[:, 1]

print('=== CALIBRATED MODEL (loaded from disk) ===')
m_test_cal = metrics_table(ytest, pc_test_loaded, 'X_test')
m_val_cal = metrics_table(yval, pc_val_loaded, 'X_val')

# Confusion matrix at business threshold (calibrated)
pred_test_cal = (pc_test_loaded >= THRESH).astype(int)
cm_cal = confusion_matrix(ytest, pred_test_cal)
tn_c, fp_c, fn_c, tp_c = cm_cal.ravel()
print()
print('Confusion matrix @ THRESH=%.3f (calibrated, X_test):' % THRESH)
print('  TN=%d  FP=%d  FN=%d  TP=%d' % (tn_c, fp_c, fn_c, tp_c))
print('  TPR=%.4f  FPR=%.4f  FNR=%.4f' % (tp_c/(tp_c+fn_c), fp_c/(fp_c+tn_c), fn_c/(fn_c+tp_c)))

# --- Side-by-side for the 3 selected points ---
print()
print('=== SIDE-BY-SIDE: raw vs calibrated (the 3 selected applications) ===')
print('%-12s %10s %10s %10s %10s' % ('Point', 'Raw P(app)', 'Cal P(app)', 'DenialRisk', 'Tier'))
for label, i in [('HIGH', hi_app), ('DENIED', hi_den), ('BORDERLINE', border)]:
    raw_p = p_test[i]
    cal_p = pc_test_loaded[i]
    print('%-12s %10.4f %10.4f %10.4f %s' % (label, raw_p, cal_p, 1-cal_p, risk_tier(1-cal_p)))

# --- Calibration curve (reliability diagram) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (y, p_raw, p_cal, name) in zip(axes, [
    (yval, p_val, pc_val_loaded, 'X_val'),
    (ytest, p_test, pc_test_loaded, 'X_test')]):
    frac_o, mean_p = calibration_curve(y, p_raw, n_bins=10, strategy='quantile')
    frac_c, mean_c = calibration_curve(y, p_cal, n_bins=10, strategy='quantile')
    ax.plot([0,1],[0,1],'k--', label='perfect')
    ax.plot(mean_p, frac_o, 'o-', label='raw')
    ax.plot(mean_c, frac_c, 's-', label='calibrated')
    ax.set_xlabel('mean predicted P(approve)')
    ax.set_ylabel('observed approval rate')
    ax.set_title('Calibration (%s)' % name)
    ax.legend()
plt.tight_layout(); plt.savefig(FIG / 'calibration_comparison.png', dpi=120); plt.close()
print('wrote', FIG / 'calibration_comparison.png')

=== CALIBRATED MODEL (loaded from disk) ===
[X_test @0.5] ROC-AUC=0.8933 PR-AUC=0.9532 P=0.8884 R=0.9367 F1=0.9119
[X_val @0.5] ROC-AUC=0.8930 PR-AUC=0.9531 P=0.8885 R=0.9365 F1=0.9119

Confusion matrix @ THRESH=0.858 (calibrated, X_test):
  TN=357861  FP=61144  FN=325778  TP=908214
  TPR=0.7360  FPR=0.1459  FNR=0.2640

=== SIDE-BY-SIDE: raw vs calibrated (the 3 selected applications) ===
Point        Raw P(app) Cal P(app) DenialRisk       Tier
HIGH             0.9999     0.9832     0.0168 Low (high approval prob)
DENIED           0.0001     0.1014     0.8986 Very high (low approval prob)
BORDERLINE       0.6366     0.8580     0.1420 Low (high approval prob)
wrote /Volumes/Mitul/Projects/home-mortage-approval-predictor/figures/final_demo/calibration_comparison.png


## **Step 5 - SHAP global interpretability**

Fits a `TreeExplainer` on the raw model over a stratified 50k-row explanation sample drawn from `X_val` (aligned positionally with `val_demographics_lookup.parquet`). Produces the beeswarm plot (per-feature distribution of impact, colored by feature value) and the bar chart (mean |SHAP| per feature = global importance).

In [6]:
N_EXPLAIN = 50_000
rng = np.random.RandomState(42)
pos = rng.choice(len(Xv), size=N_EXPLAIN, replace=False)
X_explain = Xv[pos]
demo_explain = pd.read_parquet(M / 'val_demographics_lookup.parquet').iloc[pos].reset_index(drop=True)
assert len(demo_explain) == N_EXPLAIN, 'demographics alignment failed'
print('explanation sample:', X_explain.shape, '| demographics aligned:', len(demo_explain) == N_EXPLAIN)

t0 = time.time()
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(X_explain)
if isinstance(sv, list):
    sv = sv[1]
ev = explainer.expected_value
if isinstance(ev, (list, np.ndarray)):
    ev = float(np.asarray(ev)[1]) if np.asarray(ev).size > 1 else float(np.asarray(ev).item())
print('SHAP done in %.1fs | sv shape %s | expected_value %.5f' % (time.time() - t0, sv.shape, ev))

# Global plots
plt.figure(); shap.summary_plot(sv, X_explain, feature_names=FEATURES, show=False, max_display=25)
plt.tight_layout(); plt.savefig(FIG / 'shap_beeswarm.png', dpi=120); plt.close()
plt.figure(); shap.summary_plot(sv, X_explain, plot_type='bar', feature_names=FEATURES, show=False, max_display=25)
plt.tight_layout(); plt.savefig(FIG / 'shap_bar.png', dpi=120); plt.close()
print('wrote', FIG / 'shap_beeswarm.png')
print('wrote', FIG / 'shap_bar.png')

# Top 10 global features
shap_abs = np.abs(sv).mean(0)
top_idx = np.argsort(-shap_abs)[:10]
print()
print('Top 10 features by mean |SHAP|:')
for rank, j in enumerate(top_idx, 1):
    print('  %d. %-45s %+.4f' % (rank, FEATURES[j], shap_abs[j]))

explanation sample: (50000, 78) | demographics aligned: True
SHAP done in 1080.9s | sv shape (50000, 78) | expected_value -0.00079
wrote /Volumes/Mitul/Projects/home-mortage-approval-predictor/figures/final_demo/shap_beeswarm.png
wrote /Volumes/Mitul/Projects/home-mortage-approval-predictor/figures/final_demo/shap_bar.png

Top 10 features by mean |SHAP|:
  1. loan_purpose_1                                +0.5046
  2. debt_to_income_ratio_missing                  +0.3776
  3. loan_to_value_ratio                           +0.3425
  4. income                                        +0.2675
  5. property_value                                +0.2623
  6. loan_to_income_ratio                          +0.1671
  7. tract_minority_population_percent             +0.1469
  8. lien_status_1                                 +0.1171
  9. preapproval_1                                 +0.1089
  10. co_applicant_credit_score_type_10             +0.1084


## **Step 6 - SHAP local (per-application) interpretability**

Generates SHAP waterfall plots for each of the 3 selected applications. The waterfall shows how the baseline log-odds (expected_value) is pushed step-by-step by each feature to reach the final prediction. Positive SHAP values push toward approval; negative values push toward denial.

In [8]:
print('=== Local SHAP explanations for the 3 selected applications ===')
print()
# Compute SHAP for the 3 test points directly (not from the 50k X_val sample)
X_test_3 = Xt[[hi_app, hi_den, border]]
sv_3 = explainer.shap_values(X_test_3)
if isinstance(sv_3, list):
    sv_3 = sv_3[1]

for idx, (label, i) in enumerate([('HIGH_APPROVAL', hi_app), ('DENIED_LOW', hi_den), ('BORDERLINE', border)]):
    exp_i = shap.Explanation(values=sv_3[idx], base_values=ev, data=X_test_3[idx], feature_names=FEATURES)
    plt.figure(figsize=(10, 7))
    shap.plots.waterfall(exp_i, max_display=12, show=False)
    plt.tight_layout(); plt.savefig(FIG / ('waterfall_%s.png' % label), dpi=120); plt.close()
    print('wrote', FIG / ('waterfall_%s.png' % label))
    print()
    raw = p_test[i]
    p = float(cal_loaded.predict_proba(np.array([[raw]]))[:, 1][0])
    print('--- %s: index=%d ---' % (label, i))
    print('  Calibrated P(approve) = %.4f | denial risk = %.4f | tier: %s' % (p, 1-p, risk_tier(1-p)))
    print('  Actual approved = %d' % ytest[i])
    order = np.argsort(-np.abs(sv_3[idx]))
    print('  Top SHAP factors (log-odds):')
    for j in order[:8]:
        f = FEATURES[j]; v = sv_3[idx][j]
        print('    %-45s %+.4f %s' % (f, v, '-> approval' if v > 0 else '-> denial'))
    print()

=== Local SHAP explanations for the 3 selected applications ===

wrote /Volumes/Mitul/Projects/home-mortage-approval-predictor/figures/final_demo/waterfall_HIGH_APPROVAL.png

--- HIGH_APPROVAL: index=384459 ---
  Calibrated P(approve) = 0.9832 | denial risk = 0.0168 | tier: Low (high approval prob)
  Actual approved = 1
  Top SHAP factors (log-odds):
    preapproval_1                                 +3.9623 -> approval
    preapproval_2                                 +1.6976 -> approval
    loan_purpose_1                                +0.9216 -> approval
    debt_to_income_ratio_missing                  +0.8931 -> approval
    prepayment_penalty_term                       +0.7080 -> approval
    property_value                                +0.5385 -> approval
    income                                        -0.3561 -> denial
    debt_to_income_ratio                          +0.2561 -> approval

wrote /Volumes/Mitul/Projects/home-mortage-approval-predictor/figures/final_demo/waterfa

## **Step 7 - Wrap-up summary**

Key results from this demonstration:

- **Model:** Optuna-tuned XGBoost (max_depth=13, 20-trial). Reproduced test ROC-AUC=0.8933, PR-AUC=0.9532.
- **Calibration:** Platt (logistic) scaler fit on `X_val` only, persisted to `artifacts/calibrated_model.pkl`, reloaded and verified. Raw probabilities are distorted by `scale_pos_weight`; calibrated probabilities are better calibrated and drive the business threshold.
- **Business threshold:** THRESH=0.858. At this threshold on the calibrated model, test FPR and FNR reflect the precision-first operating point.
- **3 representative applications:** high-approval (near-certain), denied (near-certain), and borderline (just above threshold) - each with calibrated probabilities and SHAP explanations.
- **SHAP:** TreeExplainer on raw model over 50k explanation sample. Global importance ranked by mean |SHAP|; local waterfall plots for each of the 3 selected applications.
- **Artifact:** calibrated model persisted at `artifacts/calibrated_model.pkl` - survives save/reload cycle without retraining (~16 min saved).
- **Scope note:** this is a decision-support demonstration. It does not implement a fairness-constrained optimizer or retrain the model.

In [ ]:
lines = [
    "# Final Demo - Home Mortgage Application Approval Classifier", "",
    "- Model: Optuna-tuned XGBoost (max_depth=13, 20-trial). Reproduced test ROC-AUC=0.8933, PR-AUC=0.9532.",
    "- Calibration: Platt (logistic) scaler fit on X_val only, persisted to artifacts/calibrated_model.pkl, reloaded and verified.",
    "- Business threshold: THRESH=0.858.",
    "- 3 representative applications: high-approval (near-certain), denied (near-certain), borderline (just above threshold).",
    "- SHAP: TreeExplainer on raw model, stratified sample n=50000 (X_val).",
    "- Calibrated model artifact: artifacts/calibrated_model.pkl (survives save/reload, ~16 min retraining saved).",
    "- Figures: figures/final_demo/",
    "- Scope note: decision-support demonstration only; no fairness-constrained optimizer or retraining.",
]
summary = chr(10).join(lines)
print(summary)
open(MARK / 'final_demo_summary.md', 'w').write(summary)
print('wrote', MARK / 'final_demo_summary.md')